In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import scanpy as sc

In [ ]:
def _edge_caption(d):
    gene = (d.get("gene") or "").strip()
    rxn = (d.get("reaction") or "").strip()
    if gene and rxn:
        return f"{gene}\n{rxn}"
    return gene or rxn or ""


def plot_pathway(g, figsize=(14, 14), seed=42, node_size=400):
    """Draw a KEGG pathway graph: compound names on nodes, gene + reaction on edges."""
    fig, ax = plt.subplots(figsize=figsize, dpi=500)
    n_nodes = max(g.number_of_nodes(), 1)
    pos = nx.spring_layout(g, seed=seed, k=1.8 / n_nodes**0.5)

    node_labels = {n: str(d.get("name", n)) for n, d in g.nodes(data=True)}
    edge_labels = {(u, v): _edge_caption(d) for u, v, d in g.edges(data=True)}

    nx.draw(
        g,
        pos,
        labels=node_labels,
        with_labels=True,
        font_size=7,
        node_size=node_size,
        node_color="lightsteelblue",
        edgecolors="steelblue",
        linewidths=0.6,
        width=0.8,
        arrowsize=12,
        node_shape="o",
        ax=ax,
    )
    nx.draw_networkx_edge_labels(
        g,
        pos,
        edge_labels=edge_labels,
        font_size=5,
        ax=ax,
        rotate=False,
        bbox={"boxstyle": "round,pad=0.2", "fc": "white", "ec": "none", "alpha": 0.85},
    )
    ax.set_axis_off()
    fig.tight_layout()
    return fig, ax

In [ ]:
from essential.kegg_pathways import list_kegg_pathways, kegg_pathway_to_graph

pathways = list_kegg_pathways("eco")
for pathway_info in pathways:
    pathway_id = pathway_info["pathway_id"]
pathways = [pathway_info for pathway_info in pathways if pathway_info["pathway_id"].startswith("eco00")]

In [ ]:
# for pathway_info in pathways:
#     pathway_id = pathway_info["pathway_id"]
#     fig, ax = plt.subplots(figsize=(10, 10))
#     g = kegg_pathway_to_graph(pathway_id, backbone=True)

#     pos = nx.spring_layout(g, seed=42)
#     edge_labels = {(u, v): d.get("gene", "") for u, v, d in g.edges(data=True)}
#     nx.draw(
#         g,
#         pos,
#         with_labels=False,
#         node_size=10,
#         node_color="steelblue",
#         ax=ax,
#     )
#     nx.draw_networkx_edge_labels(
#         g, pos, edge_labels=edge_labels, font_size=5, ax=ax, rotate=False
#     )

In [ ]:
# lps biosynthesis
# pathway_id = "eco00730"
pathway_id = "eco00540"
# pathway_id = "eco01100"
# pathway_id = "eco00010"
g = kegg_pathway_to_graph(pathway_id, backbone=True, return_multigraph=False)
plot_pathway(g, node_size=50)

In [ ]:
g2 = kegg_pathway_to_graph("eco00520", backbone=True, return_multigraph=False)
plot_pathway(g2, node_size=50)

In [ ]:
from essential.pathway_discontinuity import PathwayDiscontinuity

In [ ]:
adata = sc.read_h5ad("../data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.h5ad")
adata.X = adata.layers["reads"]
adata = adata[~adata.obs["target"].isna()].copy()
sc.pp.highly_variable_genes(adata, n_top_genes=1000, flavor="seurat_v3")
sc.pp.pca(adata, use_highly_variable=True, n_comps=50)

pda = PathwayDiscontinuity(adata, representation_obsm_key="X_pca", metabolic_graph=g2, perturbation_obs_key="target", global_sigma=5.0)
results = pda.fit(threshold=0.01, mode="mmd_stat")

In [ ]:
%matplotlib inline

In [ ]:
results.gene_pair_scores["score"].hist()
plt.show()

In [ ]:
results.plot(show_chevrons=False, dpi=1000)